# 학습 발산 추적 (v2)

지난 실행에서 **후보 12번(59.25 s)과 20번(27.55 s)**이 매번 똑같이 터졌습니다.
오류: `FactorizeHessian: rank-deficient sparse Hessian` (robot_0 몸통)

이 노트북은 그 두 후보만 설정 5가지로 다시 돌려 원인을 좁힙니다. 약 5분 걸립니다.

| 설정 | 바꾸는 것 | 이것만 안 터지면 |
|---|---|---|
| original | 없음 | – |
| implicit_kd | 관절 감쇠를 MuJoCo 쪽으로 | 앞 힌지 떨림이 원인 |
| armature | 앞 힌지 가상 관성 추가 | 앞 힌지 떨림이 원인 |
| solver_cg | 구속 계산기 Newton → CG | 계산기 수치 문제 |
| jac_dense | 희소 → 밀집 행렬 | 희소 분해 수치 문제 |

직전 물리 스텝 기록에서 `nefc`/`eq_on`(구속 수)이 갑자기 바뀌면 발 고정 구속, `qacc`가 먼저 치솟으면 몸통 위치 덮어쓰기가 의심됩니다.

`BARISimulation` 폴더에 두고 **`barisimulation` 커널**로 위에서부터 실행하세요.
(후보 목록을 지난번과 똑같이 만들기 위해 `EPISODES = 32`는 그대로 둡니다.)

In [ ]:
import dataclasses, gc, json, time
from collections import deque
from collections import Counter
import numpy as np
import mujoco

from bari_sim.policies import LinearPolicy, PolicyMetadata
from bari_sim.robot.specification import DEFAULT_ROBOT
from bari_sim.simulation import SceneRequest, Simulation
from bari_sim.tasks import parse_robot_grid, task_definition

ROBOTS = '2x5'
TASK, DIFFICULTY = 'gap', 1
EPISODES = 32           # 정책 후보 수 (학습의 population에 해당)
DURATION_S = 120.0       # 후보당 시뮬레이션 시간
SEED = 7
SCALE = 0.75            # 학습 첫 세대와 같은 탐색 폭
VARIANTS = ('original', 'implicit_kd')

grid = parse_robot_grid(ROBOTS)
task = task_definition(TASK, DIFFICULTY)
metadata = PolicyMetadata(task=task.name.value, difficulty=task.difficulty, robots=str(grid), seed=SEED)
rng = np.random.default_rng(SEED)
mean = LinearPolicy.idle(metadata).parameters()
candidates = rng.normal(mean, SCALE, size=(EPISODES, LinearPolicy.PARAMETER_COUNT))
candidates[0] = mean     # 학습과 같이 0번은 '정지' 정책

BAD = {int(mujoco.mjtWarning.mjWARN_BADQACC): 'QACC',
       int(mujoco.mjtWarning.mjWARN_BADQPOS): 'QPOS',
       int(mujoco.mjtWarning.mjWARN_BADQVEL): 'QVEL'}

def make_sim(variant):
    robot = DEFAULT_ROBOT
    if variant == 'implicit_kd':
        robot = dataclasses.replace(DEFAULT_ROBOT, joint_kd_nms_rad=0.0, turn_joint_kd_nms_rad=0.0)
    sim = Simulation(SceneRequest(grid=grid, environment=task.environment, task=task), robot=robot)
    sim.model.opt.disableflags |= int(mujoco.mjtDisableBit.mjDSBL_AUTORESET)   # 폭주 시 되돌리지 않고 멈춤
    if variant == 'implicit_kd':
        for rid in range(sim.robot_count):
            for h in range(2):
                sim.model.dof_damping[int(sim.model.jnt_dofadr[int(sim.joint_ids[rid, h])])] += DEFAULT_ROBOT.joint_kd_nms_rad
    return sim

probe = make_sim('original')
d0 = int(probe.model.jnt_dofadr[int(probe.joint_ids[0, 1])])
print('MuJoCo', mujoco.__version__, '| 원본 앞 힌지 damping =', float(probe.model.dof_damping[d0]),
      '(0.004면 원본 코드, 0.016이면 이미 패치 적용됨)')
del probe; gc.collect()

def dof_label(sim, dof):
    if dof < 0 or dof >= sim.model.nv:
        return f'dof {dof}'
    jnt = int(sim.model.dof_jntid[dof])
    return f'dof {dof} ({mujoco.mj_id2name(sim.model, mujoco.mjtObj.mjOBJ_JOINT, jnt)})'


## 원인 추적 (터진 후보만)

In [ ]:
# 원본에서 터진 후보(12, 20)만 설정 5가지로 다시 돌리고, 터지기 직전 물리 스텝을 기록
FOCUS = {20: 27.55, 12: 59.25}            # 후보 번호: 원본에서 터진 시각(초)
TRIALS = ['original', 'implicit_kd', 'armature', 'solver_cg', 'jac_dense']

def make_trial(name):
    sim = make_sim('implicit_kd' if name == 'implicit_kd' else 'original')
    m = sim.model
    if name == 'armature':
        for rid in range(sim.robot_count):
            m.dof_armature[int(m.jnt_dofadr[int(sim.joint_ids[rid, 1])])] = 2e-5
    elif name == 'solver_cg':
        m.opt.solver = mujoco.mjtSolver.mjSOL_CG
    elif name == 'jac_dense':
        m.opt.jacobian = mujoco.mjtJacobian.mjJAC_DENSE
    return sim

def trace_episode(sim, params, t_stop):
    sim.reset()
    for k in BAD:
        try: sim.data.warning[k].number = 0
        except Exception: pass
    m, d = sim.model, sim.data
    jid = mujoco.mj_name2id(m, mujoco.mjtObj.mjOBJ_JOINT, 'robot_0_root')
    dof, qadr = int(m.jnt_dofadr[jid]), int(m.jnt_qposadr[jid])
    hq = [int(m.jnt_qposadr[int(sim.joint_ids[0, h])]) for h in range(2)]
    log = deque(maxlen=15)

    def grab(_):
        eq_on = int(np.sum(d.eq_active)) if hasattr(d, 'eq_active') else -1
        log.append(dict(t=round(float(d.time), 3), ncon=int(d.ncon), nefc=int(d.nefc), eq_on=eq_on,
                        qacc=round(float(np.abs(d.qacc[dof:dof + 6]).max()), 1),
                        qvel=round(float(np.abs(d.qvel[dof:dof + 6]).max()), 3),
                        z=round(float(d.qpos[qadr + 2]), 4),
                        hinge=(round(float(d.qpos[hq[0]]), 3), round(float(d.qpos[hq[1]]), 3))))
        return None

    policy = LinearPolicy.from_parameters(params, metadata)
    obs = sim.observations()
    while sim.time_s + 1e-9 < t_stop:
        actions = {rid: policy.act(obs[rid]) for rid in range(sim.robot_count)}
        err = None
        try:
            obs = sim.step(actions, frame_callback=grab, render_hz=1.0 / sim.timestep_s).observations
        except Exception as exc:
            err = f'{type(exc).__name__}: {exc}'
        bad = any(sim.data.warning[k].number for k in BAD)
        finite = np.isfinite(sim.data.qpos).all() and np.isfinite(sim.data.qvel).all()
        if err or bad or not finite:
            return 'UNSTABLE', round(sim.time_s, 2), err, actions, list(log)
    return 'ok', round(sim.time_s, 2), None, None, None

trace_results = {}
for name in TRIALS:
    sim = make_trial(name)
    for idx, t_fail in FOCUS.items():
        t0 = time.perf_counter()
        st, t, err, acts, log = trace_episode(sim, candidates[idx], t_fail + 1.0)
        trace_results[(name, idx)] = st
        print(f'[{name:11s} 후보 {idx}] {st:8s} t={t:6.2f}s ({time.perf_counter() - t0:.0f}s)  {err or ""}', flush=True)
        if st != 'ok':
            print('   로봇별 직전 동작:', {r: (a.motion.name, a.lift.name, a.grip.name) for r, a in acts.items()})
            print('   robot_0 직전 물리 스텝 (0.002 s 간격):')
            for row in log:
                print('    ', row)
    del sim; gc.collect()

print('\n요약 (UNSTABLE = 터짐):')
for name in TRIALS:
    print(f'  {name:11s}', {idx: trace_results[(name, idx)] for idx in FOCUS})


## (선택) 전체 비교
`RUN_FULL = True`로 바꾸면 32개 후보를 두 설정으로 모두 돌립니다. 오래 걸립니다.

In [ ]:
RUN_FULL = False   # True로 바꾸면 32개 후보 × 2설정 전체 비교(수십 분 걸림)

def episode(sim, params):
    sim.reset()
    for k in BAD:
        try: sim.data.warning[k].number = 0
        except Exception: pass
    policy = LinearPolicy.from_parameters(params, metadata)
    obs = sim.observations()
    counts = Counter()
    last = {}
    t0 = time.perf_counter()
    while sim.time_s + 1e-9 < DURATION_S:
        actions = {rid: policy.act(obs[rid]) for rid in range(sim.robot_count)}
        for a in actions.values():
            counts[f'motion.{a.motion.name}'] += 1; counts[f'lift.{a.lift.name}'] += 1; counts[f'grip.{a.grip.name}'] += 1
        attaching_before = [rid for rid in range(sim.robot_count) if sim.attachments.is_attaching(rid)]
        error = None
        try:
            result = sim.step(actions)
            obs = result.observations
        except Exception as exc:
            error = f'{type(exc).__name__}: {exc}'
        hits = {name: (sim.data.warning[k].number, sim.data.warning[k].lastinfo) for k, name in BAD.items()
                if sim.data.warning[k].number > 0}
        finite = np.isfinite(sim.data.qpos).all() and np.isfinite(sim.data.qvel).all()
        if hits or error or not finite:
            where = [f'{name}@{dof_label(sim, info)}' for name, (n, info) in hits.items()]
            robots = sorted({int(lab.split('robot_')[1].split('_')[0]) for lab in where if 'robot_' in lab})
            return dict(status='UNSTABLE', t=round(sim.time_s, 2), where=where, error=error,
                        actions_before={rid: (actions[rid].motion.name, actions[rid].lift.name, actions[rid].grip.name)
                                        for rid in (robots or range(sim.robot_count))},
                        attaching_before=attaching_before, counts=counts, wall=time.perf_counter() - t0)
        if sim.evaluator is not None and sim.evaluator.complete:
            break
    res = sim.evaluator.result() if sim.evaluator is not None else None
    return dict(status='ok', t=round(sim.time_s, 2), score=None if res is None else res.metrics.get('score'),
                counts=counts, wall=time.perf_counter() - t0)

results = {}
for variant in (VARIANTS if RUN_FULL else ()):
    sim = make_sim(variant)
    results[variant] = []
    for i, params in enumerate(candidates):
        r = episode(sim, params)
        results[variant].append(r)
        grip = {k.split('.')[1]: v for k, v in r['counts'].items() if k.startswith('grip.')}
        line = f"[{variant:11s} 후보 {i:2d}] {r['status']:8s} t={r['t']:6.2f}s  ({r['wall']:.0f}s)  grip={grip}"
        if r['status'] != 'ok':
            line += f"\n      위치: {r['where']}  오류: {r['error']}\n      직전 동작: {r['actions_before']}\n      직전 결합 중 로봇: {r['attaching_before']}"
        print(line, flush=True)
    del sim; gc.collect()


In [ ]:
if RUN_FULL:
    for variant in VARIANTS:
        rs = results.get(variant, [])
        bad = [r for r in rs if r['status'] != 'ok']
        locs = Counter(w.split('@')[1] for r in bad for w in r['where'])
        with_attach = sum(1 for r in bad if r['attaching_before'] or any(a[2] == 'ATTACH' for a in r['actions_before'].values()))
        print(f"{variant:11s}: 폭주 {len(bad)}/{len(rs)}  | ATTACH/결합 관련 {with_attach}  | 위치 {dict(locs)}")
    paired = [(i, a['status'], b['status']) for i, (a, b) in enumerate(zip(results.get('original', []), results.get('implicit_kd', [])))]
    print('후보별 (원본, implicit_kd):', paired)
